# LANGKAH 1 : Data Prepocessing

In [1]:
# Install library yang dibutuhakan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tabulate import tabulate
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set tampilan
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Semua library berhasil diimport!")


✅ Semua library berhasil diimport!


In [131]:
# Load dataset 

df = pd.read_csv('dataset.csv',encoding='cp1252')
# df = df.dropna(axis=1, how='all') # Menghapus kolom yang kosong

# Cek data yang tersedia pada dataset

print("=" * 60)
print("Informasi Dataset :")
print("=" * 60)
print(f"Jumlah baris: {df.shape[0]}")
print(f"Jumlah kolom: {df.shape[1]}")

print(f"Nama kolom: {df.columns.tolist()}")
print("\n" + "=" * 60)




print("5 Data Pertama :")

print(tabulate(df.head(), headers='keys', tablefmt='psql', showindex=False))



Informasi Dataset :
Jumlah baris: 1043
Jumlah kolom: 26
Nama kolom: ['No', 'Nama Lengkap', 'Prodi', 'Jenis Kelamin', 'Jarak Tempat Tinggal kekampus (Km)', 'Asal Sekolah', 'Tahun Lulus', 'SKS', 'Ikut Organisasi', 'Ikut UKM', 'IPK', 'Pekerjaan Orang Tua', 'Penghasilan', 'Tanggungan', 'Status Beasiswa', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25']

5 Data Pertama :
+------+--------------------------+-----------------------------------+-----------------+--------------------------------------+--------------------------+---------------+-------+-------------------+------------+-------+-----------------------+---------------+--------------+-------------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+
|   No | Nama Lengkap             | Prodi  

In [132]:
# Cek dulu apakah ada data yag kosong

print("=" * 60)
print("🔍 CEK MISSING VALUES")
print("=" * 60)
print(df.isnull().sum())

print("\n" + "=" * 60)
print("📝 DATA TYPES")
print("=" * 60)
print(df.dtypes)

🔍 CEK MISSING VALUES
No                                       1
Nama Lengkap                             1
Prodi                                    1
Jenis Kelamin                            1
Jarak Tempat Tinggal kekampus (Km)       1
Asal Sekolah                             1
Tahun Lulus                              1
SKS                                      1
Ikut Organisasi                          1
Ikut UKM                                 1
IPK                                      1
Pekerjaan Orang Tua                      1
Penghasilan                              1
Tanggungan                               1
Status Beasiswa                          1
Unnamed: 15                           1043
Unnamed: 16                           1043
Unnamed: 17                           1043
Unnamed: 18                           1043
Unnamed: 19                           1043
Unnamed: 20                           1043
Unnamed: 21                           1043
Unnamed: 22                      

In [133]:
# Data Cleaning


df_clean = df.copy()
df_clean = df_clean.dropna(axis=1, how='all')


df_clean.columns = df_clean.columns.str.strip()


print(f"Baris duplikat sebelum cleaning: {df_clean.duplicated().sum()}")
df_clean = df_clean.drop_duplicates()
print(f"Baris duplikat setelah cleaning: {df_clean.duplicated().sum()}")


for col in df_clean.columns:
    if df_clean[col].isnull().sum() > 0:
        if df_clean[col].dtype in ['int64', 'float64']:
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        else:
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print("\n✅ Data cleaning selesai!")
print(f"Shape setelah cleaning: {df_clean.shape}")

print(tabulate(df_clean.head(), headers='keys', tablefmt='psql', showindex=False))


Baris duplikat sebelum cleaning: 0
Baris duplikat setelah cleaning: 0

✅ Data cleaning selesai!
Shape setelah cleaning: (1043, 15)
+------+--------------------------+-----------------------------------+-----------------+--------------------------------------+--------------------------+---------------+-------+-------------------+------------+-------+-----------------------+---------------+--------------+-------------------+
|   No | Nama Lengkap             | Prodi                             | Jenis Kelamin   | Jarak Tempat Tinggal kekampus (Km)   | Asal Sekolah             |   Tahun Lulus |   SKS | Ikut Organisasi   | Ikut UKM   |   IPK | Pekerjaan Orang Tua   | Penghasilan   |   Tanggungan | Status Beasiswa   |
|------+--------------------------+-----------------------------------+-----------------+--------------------------------------+--------------------------+---------------+-------+-------------------+------------+-------+-----------------------+---------------+--------------+--

1043 baris (record/responden) 15 kolom (turun dari 26 kolom asli — 15 kolom asli + 11 kolom Unnamed yang udah kehapus)

In [135]:
# Pisah ground truth ke kolom terpisah

ground_truth_col = 'Status Beasiswa'

df_clean['Ground_Truth'] = df_clean[ground_truth_col].map({'Terima': 1, 'Tidak': 0})


print(df_clean['Ground_Truth'].value_counts())
print(f"\nTotal: {df_clean['Ground_Truth'].sum()} Menerima, {(df_clean['Ground_Truth']==0).sum()} Tidak Menerima")

fitur_kolom = [col for col in df_clean.columns 
               if col not in ['No', 'Nama Lengkap', 'Asal Sekolah', 
                             ground_truth_col, 'Ground_Truth', 'Prodi']]

print(f"\n📋 Fitur yang digunakan untuk clustering: {fitur_kolom}")



Ground_Truth
0    771
1    272
Name: count, dtype: int64

Total: 272 Menerima, 771 Tidak Menerima

📋 Fitur yang digunakan untuk clustering: ['Jenis Kelamin', 'Jarak Tempat Tinggal kekampus (Km)', 'Tahun Lulus', 'SKS', 'Ikut Organisasi', 'Ikut UKM', 'IPK', 'Pekerjaan Orang Tua', 'Penghasilan', 'Tanggungan']


In [137]:
df_fitur = df_clean[fitur_kolom].copy()

le_dict = {}

for col in df_fitur.columns:
    if not pd.api.types.is_numeric_dtype(df_fitur[col]):
        le = LabelEncoder()
        df_fitur[col] = le.fit_transform(df_fitur[col].astype(str))
        le_dict[col] = le
        print(f"Encoded '{col}': {dict(zip(le.classes_, range(len(le.classes_))))}")

print("Data setelah di encoding: ")
df_fitur.head()

Encoded 'Jenis Kelamin': {'L': 0, 'P': 1}
Encoded 'Jarak Tempat Tinggal kekampus (Km)': {'Dekat': 0, 'Jauh': 1}
Encoded 'Ikut Organisasi': {'Ikut': 0, 'Tidak': 1}
Encoded 'Ikut UKM': {'Ikut': 0, 'Tidak': 1}
Encoded 'Pekerjaan Orang Tua': {'Account Coordinator': 0, 'Account Executive': 1, 'Account Representative I': 2, 'Account Representative II': 3, 'Account Representative IV': 4, 'Accountant II': 5, 'Accountant III': 6, 'Accountant IV': 7, 'Accounting Assistant I': 8, 'Accounting Assistant II': 9, 'Accounting Assistant III': 10, 'Accounting Assistant IV': 11, 'Actuary': 12, 'Administrative Assistant I': 13, 'Administrative Assistant IV': 14, 'Administrative Officer': 15, 'Analog Circuit Design manager': 16, 'Analyst Programmer': 17, 'Assistant Manager': 18, 'Assistant Media Planner': 19, 'Assistant Professor': 20, 'Associate Professor': 21, 'Automation Specialist III': 22, 'Automation Specialist IV': 23, 'Biostatistician I': 24, 'Biostatistician III': 25, 'Biostatistician IV': 26, 'Bu

,Jenis Kelamin,Jarak Tempat Tinggal kekampus (Km),Tahun Lulus,SKS,Ikut Organisasi,Ikut UKM,IPK,Pekerjaan Orang Tua,Penghasilan,Tanggungan
0,0,0,2020.0,21.0,0,0,3.57,169,1,4.0
1,0,0,2020.0,21.0,1,0,2.95,30,1,2.0
2,1,0,2020.0,21.0,1,0,3.67,104,1,4.0
3,0,0,2020.0,21.0,1,0,3.19,169,2,2.0
4,1,1,2020.0,21.0,1,0,3.19,169,1,2.0


In [139]:
# Scaling data numerik

scaler = StandardScaler()  

df_scaled = pd.DataFrame(
    scaler.fit_transform(df_fitur),
    columns=df_fitur.columns,
    index=df_fitur.index
)

print("Data setelah scaling")
# print(df_scaled.describe().round(3))

print(tabulate(df_scaled.describe().round(3),headers='keys', tablefmt='psql', showindex=False))

print("Scaling selesai! Ground Truth tidak termasuk dalam scaling.")

Data setelah scaling
+-----------------+--------------------------------------+---------------+----------+-------------------+------------+----------+-----------------------+---------------+--------------+
|   Jenis Kelamin |   Jarak Tempat Tinggal kekampus (Km) |   Tahun Lulus |      SKS |   Ikut Organisasi |   Ikut UKM |      IPK |   Pekerjaan Orang Tua |   Penghasilan |   Tanggungan |
|-----------------+--------------------------------------+---------------+----------+-------------------+------------+----------+-----------------------+---------------+--------------|
|        1043     |                             1043     |      1043     | 1043     |          1043     |   1043     | 1043     |              1043     |      1043     |     1043     |
|           0     |                                0     |        -0     |    0     |             0     |     -0     |   -0     |                -0     |         0     |        0     |
|           1     |                                1  